# Prompt 11: Spark SQL Query Optimization and Execution Plans
## Databricks Certified Associate Developer for Apache Spark
### Topic 2 — Using Spark SQL (20%)

---

## The Catalyst Optimizer Pipeline

Catalyst is Spark's built-in query optimizer. Every DataFrame/SQL query passes through
4 phases before a physical plan is executed:

```
User Code (DataFrame API or spark.sql())
          │
          ▼
┌─────────────────────┐
│  1. ANALYSIS        │  ← Resolve column names, types, functions against catalog
│  Unresolved Plan    │
│  → Resolved Plan    │
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  2. LOGICAL         │  ← Rule-based optimizations (predicate pushdown,
│  OPTIMIZATION       │    column pruning, constant folding, etc.)
│  Resolved Plan      │
│  → Optimized Plan   │
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  3. PHYSICAL        │  ← Generate multiple physical plans, use cost model
│  PLANNING           │    to pick best (BroadcastHashJoin vs SortMergeJoin, etc.)
│  Optimized Plan     │
│  → Physical Plan    │
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  4. CODE            │  ← Whole-stage code generation (WholeStageCodeGen)
│  GENERATION         │    compiles plan to JVM bytecode
│  Physical Plan      │
│  → JVM Bytecode     │
└─────────┬───────────┘
          │
          ▼
       EXECUTE
```

---

## Key Optimization Rules Applied in Phase 2

| Optimization | What it does | Benefit |
|-------------|-------------|--------|
| **Predicate Pushdown** | Moves `filter`/`WHERE` clauses as early as possible in the plan (closer to data source) | Read less data |
| **Column Pruning** | Eliminates columns not needed downstream (projection pushdown) | Reduced I/O and memory |
| **Constant Folding** | Pre-evaluates constant expressions at plan time: `1 + 1` → `2` | Avoids runtime computation |
| **Join Reordering** | Reorders joins to put smaller tables first | Less intermediate data |
| **Broadcast Join** | Replaces sort-merge join with broadcast join when one side is small | Eliminates shuffle |
| **Partition Pruning** | Skips reading partitions that don't match filter conditions | Reads only needed files |
| **Subquery Elimination** | Rewrites correlated subqueries as joins | More efficient execution |
| **Null Propagation** | Simplifies null-related expressions statically | Faster evaluation |

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, broadcast

spark = SparkSession.builder \
    .appName('QueryOptimization') \
    .master('local[*]') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Large fact table (orders)
orders = spark.createDataFrame([
    (1,  101, 'Electronics', 299.99,  '2024-01-05', 'US'),
    (2,  102, 'Clothing',     49.99,  '2024-01-06', 'UK'),
    (3,  101, 'Electronics', 199.99,  '2024-01-07', 'US'),
    (4,  103, 'Food',         15.50,  '2024-01-08', 'DE'),
    (5,  102, 'Clothing',     89.99,  '2024-01-09', 'UK'),
    (6,  104, 'Electronics', 499.99,  '2024-01-10', 'US'),
    (7,  101, 'Food',         22.00,  '2024-01-11', 'US'),
    (8,  105, 'Clothing',     35.00,  '2024-01-12', 'FR'),
], ['order_id', 'customer_id', 'category', 'amount', 'order_date', 'country'])

# Small dimension table (customers)
customers = spark.createDataFrame([
    (101, 'Alice',  'Gold'),
    (102, 'Bob',    'Silver'),
    (103, 'Carol',  'Bronze'),
    (104, 'Dave',   'Gold'),
    (105, 'Eve',    'Silver'),
], ['customer_id', 'name', 'tier'])

orders.createOrReplaceTempView('orders')
customers.createOrReplaceTempView('customers')

print('Orders:')
orders.show()
print('Customers:')
customers.show()

## df.explain() — Reading Execution Plans

`df.explain(mode)` prints the physical/logical execution plan to the console.

| Mode | What it shows |
|------|---------------|
| `explain()` / `explain(False)` | Physical plan only |
| `explain(True)` | Parsed, Analyzed, Optimized, and Physical plans |
| `explain('simple')` | Physical plan only |
| `explain('extended')` | All 4 plans (same as `True`) |
| `explain('codegen')` | Physical plan + generated Java code |
| `explain('cost')` | Optimized plan with cost statistics |
| `explain('formatted')` | Two-section split: plan + node details |

**How to read the plan (bottom-up):**
Plans are read from **bottom to top** — leaf nodes at the bottom are data sources; root at the top is the final output.

```
== Physical Plan ==
*(2) HashAggregate(keys=[department], functions=[avg(salary)])  ← Step 3: aggregate
+- Exchange hashpartitioning(department, 200)                   ← Step 2: shuffle
   +- *(1) HashAggregate(keys=[department], functions=[...])    ← Step 1: partial agg
      +- *(1) Filter (salary > 70000)                           ← pushed down filter
         +- *(1) Scan ...                                       ← Step 0: read data
```

**Key symbols:**
- `*(n)` — WholeStageCodeGen stage n (compiled to JVM bytecode)
- `Exchange` — shuffle (data movement between executors)
- `BroadcastExchange` — small table broadcast (no shuffle for that side)
- `+- ` — child operator

In [ ]:
# Cell 2: df.explain() — all modes
df = orders.filter(col('amount') > 100) \
           .select('order_id', 'customer_id', 'category', 'amount')

print('=== explain() — physical plan only ===')
df.explain()

print('\n=== explain("extended") — all 4 plans ===')
df.explain('extended')

print('\n=== explain("formatted") — plan + node details ===')
df.explain('formatted')

## Predicate Pushdown and Column Pruning

### Predicate Pushdown
Catalyst moves `filter`/`WHERE` conditions as early as possible — ideally into the data source
scan (e.g., Parquet, Delta Lake, JDBC). This means data is filtered before being read into memory.

```
WITHOUT pushdown:  Read ALL rows → shuffle → filter
WITH pushdown:     Filter at source → read only matching rows → (no wasted I/O)
```

### Column Pruning
Only columns referenced in the query are read from the data source.
For columnar formats (Parquet, ORC), this means unneeded columns are never read from disk.

```
orders has 6 columns: order_id, customer_id, category, amount, order_date, country
SELECT order_id, amount FROM orders WHERE amount > 100
→ Catalyst reads only order_id and amount columns from Parquet files
```

In [ ]:
# Cell 3: Predicate Pushdown and Column Pruning — observe in plan

# Without filter vs with filter — note how filter appears in the plan
df_all    = orders.select('order_id', 'amount', 'category')
df_filter = orders.filter(col('amount') > 100).select('order_id', 'amount')

print('=== Plan WITHOUT filter ===')
df_all.explain()

print('=== Plan WITH filter (note "Filter" in scan) ===')
df_filter.explain()

# Observe column pruning — only order_id and amount in the Output
print('=== Column pruning — only selected columns appear in scan output ===')
df_filter.explain('formatted')

# Note: PushedFilters appears in the scan node when reading Parquet/Delta/JDBC
# For in-memory DataFrames, the filter node appears just above the scan

# Predicate pushdown in SQL
print('=== SQL predicate pushdown — filter pushed to scan ===')
spark.sql("""
    SELECT order_id, amount
    FROM orders
    WHERE amount > 100
      AND category = 'Electronics'
""").explain()

## Join Strategies: BroadcastHashJoin vs SortMergeJoin

### BroadcastHashJoin (BHJ)
- One table is small enough to fit in driver/executor memory
- Spark broadcasts the small table to ALL executors
- Large table is NOT shuffled — each partition joins locally with the broadcast copy
- **No shuffle for the large table** → much faster
- Triggered automatically when small table < `spark.sql.autoBroadcastJoinThreshold` (default: 10 MB)
- Force with `broadcast(df)` hint

```
BroadcastHashJoin:
  Large table (NOT shuffled)     Small table (broadcast to all)
  ┌──────┐                       ┌──────┐
  │ p1   │ ◄── broadcast copy    │      │  → BroadcastExchange in plan
  │ p2   │ ◄── broadcast copy    │  4KB │  → sent to every executor
  │ p3   │ ◄── broadcast copy    └──────┘
  └──────┘
  Executors join locally — NO shuffle of large table
```

### SortMergeJoin (SMJ)
- Both tables are large (neither fits in memory for broadcast)
- Both tables are **shuffled** by join key, then **sorted** within each partition, then **merged**
- Requires 2 shuffles + 2 sorts → expensive but scalable
- Default when BHJ threshold not met

```
SortMergeJoin:
  Table A                Table B
  ┌──────┐               ┌──────┐
  │ rows │ → SHUFFLE ──► │ rows │ → co-partition by join key
  └──────┘    ↕sort      └──────┘    ↕sort
              └── merge matching keys ──┘
  Expensive: 2 shuffles + 2 sorts across both datasets
```

### Other Join Strategies

| Strategy | When used | Plan keyword |
|----------|-----------|-------------|
| BroadcastHashJoin | Small table < threshold | `BroadcastHashJoin` |
| SortMergeJoin | Both tables large | `SortMergeJoin` |
| ShuffledHashJoin | One side small-ish but > broadcast threshold | `ShuffledHashJoin` |
| BroadcastNestedLoopJoin | Non-equi joins (theta joins) | `BroadcastNestedLoopJoin` |
| CartesianProduct | Cross join (no condition) | `CartesianProduct` |

In [ ]:
# Cell 4: BroadcastHashJoin vs SortMergeJoin

# Default join — Spark decides (likely BHJ since customers is tiny)
joined_default = orders.join(customers, on='customer_id', how='inner')
print('=== Default join plan (Spark chooses strategy) ===')
joined_default.explain()

# Explicit broadcast hint
joined_broadcast = orders.join(broadcast(customers), on='customer_id', how='inner')
print('=== Explicit broadcast() hint ===')
joined_broadcast.explain()

# Force SortMergeJoin by disabling broadcast threshold
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')  # disable broadcast
joined_smj = orders.join(customers, on='customer_id', how='inner')
print('=== SortMergeJoin (broadcast disabled) ===')
joined_smj.explain()

# Restore
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))

# SQL BROADCAST hint
print('=== SQL /*+ BROADCAST(customers) */ hint ===')
spark.sql("""
    SELECT /*+ BROADCAST(customers) */
        o.order_id, o.amount, c.name, c.tier
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
""").explain()

## Adaptive Query Execution (AQE)

AQE (Spark 3.0+, **enabled by default in Spark 3.2+**) re-optimizes the physical plan
**at runtime** based on actual statistics collected during execution.

### Three Main AQE Features

| Feature | Problem solved | How it works |
|---------|---------------|-------------|
| **Coalescing shuffle partitions** | Too many small post-shuffle partitions (slow due to task overhead) | Merges small adjacent partitions after shuffle based on actual data size |
| **Dynamic partition pruning** | Scanning unneeded partitioned data | Broadcasts small side of join as a filter for the other side's scan |
| **Converting sort-merge join to broadcast join** | SMJ chosen at planning time but one side turns out small at runtime | Switches to BHJ after collecting runtime stats on actual partition sizes |

### Controlling AQE

```python
# Enable (default in Spark 3.2+)
spark.conf.set('spark.sql.adaptive.enabled', 'true')

# Target partition size after coalescing (default: 64MB)
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64m')

# Min partitions to keep (won't go below this)
spark.conf.set('spark.sql.adaptive.coalescePartitions.minPartitionNum', '1')

# Threshold for AQE to switch SMJ → BHJ at runtime (default: 30MB)
spark.conf.set('spark.sql.adaptive.autoBroadcastJoinThreshold', '30m')
```

**Exam note:** With AQE enabled, the plan shown by `explain()` before execution may differ
from the plan actually executed. The final plan is only visible in the Spark UI after the job runs.
Look for `AdaptiveSparkPlan` as the root node in `explain()` output when AQE is active.

In [ ]:
# Cell 5: AQE — enabling and observing

# Enable AQE
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')  # default

# With AQE, the plan root becomes AdaptiveSparkPlan
agg = orders.groupBy('category').agg(F.sum('amount').alias('total'))

print('=== explain() with AQE enabled — note AdaptiveSparkPlan at root ===')
agg.explain()

print('\n=== explain("extended") with AQE — shows initial plan before adaptation ===')
agg.explain('extended')

# AQE coalesce: 200 shuffle partitions → automatically coalesced to fewer
print('\n=== Run action — AQE coalesces 200 partitions down to actual data size ===')
agg.show()
# With tiny data, AQE may coalesce to 1 partition
# Check Spark UI → SQL tab → plan after execution for actual runtime plan

print('\n=== Check AQE config values ===')
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Advisory partition size: {spark.conf.get('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64m')}")

In [ ]:
# Cell 6: Partition Pruning
# Partition pruning skips reading entire partitions based on filter conditions
# Works on both static (at planning time) and dynamic (at runtime via AQE) pruning

import tempfile, os

# Write partitioned Parquet to temp dir
tmp = tempfile.mkdtemp()
parquet_path = os.path.join(tmp, 'orders_partitioned')

orders.write.partitionBy('country').mode('overwrite').parquet(parquet_path)

# Read back and filter on partition column
orders_p = spark.read.parquet(parquet_path)

print('=== All countries ===')
orders_p.explain('formatted')

print('\n=== Filter on PARTITION column (country) — PartitionFilters pushed ===')
orders_us = orders_p.filter(col('country') == 'US')
orders_us.explain('formatted')
# Look for: PartitionFilters: [isnotnull(country#xx), (country#xx = US)]
# This means Spark reads only the country=US/ partition directory

print('\n=== Filter on NON-PARTITION column (amount) — DataFilters only ===')
orders_big = orders_p.filter(col('amount') > 200)
orders_big.explain('formatted')
# Look for: DataFilters: [isnotnull(amount#xx), (amount#xx > 200.0)]
# No PartitionFilters → all partitions scanned, filter applied to each row

print('\n=== Results: US orders only (partition pruned) ===')
orders_us.show()

In [ ]:
# Cell 7: Constant Folding and other logical optimizations

# Constant folding: Catalyst pre-evaluates constant expressions
from pyspark.sql.functions import lit

# These constant expressions are folded at plan time
df_const = orders.withColumn('tax_rate',  lit(0.1) * lit(1))  # → 0.1
df_const = df_const.withColumn('tax',     col('amount') * lit(0.1 * 1))  # folded
df_const = df_const.withColumn('always_true', lit(1) == lit(1))  # → true
df_const = df_const.withColumn('never_true',  lit(1) == lit(2))  # → false

print('=== Constant folding in plan ===')
df_const.select('amount', 'tax', 'always_true', 'never_true').explain('extended')
# In optimized plan, constant expressions are replaced with their pre-computed values

# Filter on always-false condition — Catalyst eliminates the scan entirely
impossible = orders.filter(lit(1) == lit(2))
print('=== Always-false filter — plan shows LocalRelation (no scan) ===')
impossible.explain()
# Expected: LocalRelation or empty scan — Catalyst knows no rows can match

print('Count of impossible result:', impossible.count())  # 0

spark.stop()
print('Done.')

## Execution Plan Quick Reference

### explain() Nodes to Recognise

| Plan node | Meaning |
|-----------|--------|
| `FileScan parquet` / `FileScan csv` | Reading from file source |
| `Filter (...)` | Row filter (WHERE clause) |
| `Project [col1, col2]` | Column selection (SELECT) |
| `Exchange hashpartitioning(key, N)` | Shuffle — repartition by hash of key into N partitions |
| `Exchange RoundRobinPartitioning(N)` | Shuffle — `repartition(N)` without key |
| `Exchange SinglePartition` | Coalesce to 1 partition |
| `HashAggregate` | Aggregation using hash table (groupBy+agg) |
| `SortAggregate` | Aggregation via sort (used when memory exceeded) |
| `Sort [col ASC]` | Sorting |
| `BroadcastHashJoin` | Join where one side is broadcast |
| `SortMergeJoin` | Join where both sides are shuffled+sorted |
| `BroadcastExchange` | Small table being broadcast to all executors |
| `Window` | Window function computation |
| `AdaptiveSparkPlan` | Root node when AQE is enabled |
| `*(n)` prefix | WholeStageCodeGen stage n |
| `+- ` | Child operator (plan is read bottom to top) |
| `PushedFilters: [...]` | Filters pushed down into the data source |
| `PartitionFilters: [...]` | Partition pruning filters |
| `DataFilters: [...]` | Row-level filters after reading partition |
| `ReadSchema: struct<...>` | Column pruning — only these columns are read |

### Key Config Properties

| Property | Default | Purpose |
|----------|---------|--------|
| `spark.sql.adaptive.enabled` | `true` (3.2+) | Enable AQE |
| `spark.sql.autoBroadcastJoinThreshold` | 10 MB | Max size for auto broadcast join; set to `-1` to disable |
| `spark.sql.shuffle.partitions` | 200 | Default number of shuffle partitions |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | 64 MB | Target partition size after AQE coalesce |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | 30 MB | AQE threshold for SMJ → BHJ conversion at runtime |

## Real-World Scenarios

<details>
<summary>Scenario 1: Diagnosing a slow join with explain() — switching to BroadcastHashJoin</summary>

**Situation:**
A data engineer joins a 500 GB transaction table with a 2 MB currency lookup table.
The job takes 45 minutes. `explain()` shows `SortMergeJoin` in the plan.

**Diagnosis:**
The currency table (2 MB) is well below the default 10 MB broadcast threshold, but Spark
can't determine its size statically (e.g., it was computed by a subquery). AQE is disabled.

**Fix:**
```python
from pyspark.sql.functions import broadcast

# Option 1: explicit broadcast hint
result = transactions.join(broadcast(currency), on='currency_code', how='left')

# Option 2: raise the threshold
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(50 * 1024 * 1024))  # 50 MB

# Option 3: enable AQE (converts SMJ to BHJ at runtime if one side is small)
spark.conf.set('spark.sql.adaptive.enabled', 'true')

# Verify
result.explain()
# Before fix: SortMergeJoin [...]
# After fix:  BroadcastHashJoin [...] with BroadcastExchange on currency side
```

**Expected Outcome:**
Job time drops from 45 minutes to ~5 minutes. The 500 GB table is never shuffled.
The 2 MB table is broadcast once to each executor.

**Exam Sub-topic:** BroadcastHashJoin vs SortMergeJoin; `broadcast()` hint; `autoBroadcastJoinThreshold`
</details>

<details>
<summary>Scenario 2: Using explain() to verify predicate pushdown is working</summary>

**Situation:**
A developer reads a large Parquet table, applies several transformations, then filters
near the end of the pipeline. They want to confirm Catalyst is pushing the filter
to the scan and not reading all rows first.

**Code:**
```python
# Pipeline: complex transformations then filter
df = spark.read.parquet('/data/events/')
result = df.withColumn('full_name', F.concat('first_name', F.lit(' '), 'last_name')) \
           .withColumn('age_group', F.when(F.col('age') < 30, 'young').otherwise('senior')) \
           .filter(F.col('country') == 'US') \
           .select('full_name', 'age_group', 'event_type')

result.explain('formatted')
# In the FileScan node, look for:
# PushedFilters: [IsNotNull(country), EqualTo(country,US)]
# → Filter pushed down to source, only US rows read from disk

# ReadSchema shows column pruning:
# ReadSchema: struct<first_name:string,last_name:string,age:int,country:string,event_type:string>
# → Only 5 of perhaps 50 columns are read from Parquet
```

**Expected Outcome:**
`PushedFilters` in the `FileScan` node confirms the filter reaches the data source.
`ReadSchema` confirms only required columns are read.

**Exam Sub-topic:** Predicate pushdown; column pruning; reading `explain('formatted')` output
</details>

<details>
<summary>Scenario 3: AQE reduces 200 shuffle partitions to 5 after groupBy on small data</summary>

**Situation:**
A developer runs a `groupBy().agg()` on a 10 MB dataset. With `spark.sql.shuffle.partitions=200`,
the job creates 200 tiny output partitions (each ~50 KB), causing massive task scheduling overhead.

**Without AQE:**
```python
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')

result = df.groupBy('category').sum('amount')
result.cache().count()  # trigger execution

print(result.rdd.getNumPartitions())  # 200 — tiny partitions!
```

**With AQE:**
```python
spark.conf.set('spark.sql.adaptive.enabled', 'true')

result_aqe = df.groupBy('category').sum('amount')
result_aqe.cache().count()

print(result_aqe.rdd.getNumPartitions())  # 5 — AQE coalesced!
# AQE measured actual partition sizes and merged tiny ones
```

**Expected Outcome:**
AQE coalesces 200 small partitions into ~5 larger ones based on actual data size.
Task scheduling overhead drops by 97%. No manual tuning needed.

**Exam Sub-topic:** AQE coalescing shuffle partitions; `spark.sql.adaptive.enabled`
</details>

<details>
<summary>Scenario 4: Partition pruning on a Delta/Parquet table — country filter</summary>

**Situation:**
A 1 TB orders table is partitioned by `country` (50 countries). A query filters for
`country = 'US'`. Without partition pruning, all 50 partition directories are scanned.

**Code:**
```python
# Table partitioned by country
orders = spark.read.parquet('/data/orders_partitioned/')  # partitionBy('country')

# Filter on partition column
us_orders = orders.filter(col('country') == 'US')
us_orders.explain('formatted')

# Plan shows:
# PartitionFilters: [isnotnull(country#xx), (country#xx = US)]
# → Only country=US/ directory is read (1/50 of data = 20 GB instead of 1 TB)

# Dynamic Partition Pruning (DPP) with AQE:
# When joining fact table to dimension and filtering on dimension:
fact = spark.read.parquet('/data/orders_partitioned/')
dim  = spark.read.parquet('/data/countries/').filter(col('region') == 'Americas')

# AQE broadcasts the filtered dim result as a bloom filter on the fact scan
# → fact table only reads partitions matching countries in Americas
result = fact.join(dim, on='country', how='inner')
result.explain()  # shows SubqueryBroadcast or DynamicPruning nodes
```

**Expected Outcome:**
Static pruning cuts the scan from 1 TB to 20 GB (1 of 50 partitions).
Dynamic pruning (AQE) further refines the set of partitions based on runtime join results.

**Exam Sub-topic:** Partition pruning; `PartitionFilters` vs `DataFilters`; AQE dynamic partition pruning
</details>

<details>
<summary>Scenario 5: Constant folding eliminates a dead branch in a CASE WHEN expression</summary>

**Situation:**
A developer uses a feature flag stored as a config literal to toggle logic.
The flag is always `True` in production. Catalyst should eliminate the `False` branch entirely.

**Code:**
```python
from pyspark.sql.functions import when, lit, col

ENABLE_V2_LOGIC = True  # always True in production

result = df.withColumn('price',
    when(lit(ENABLE_V2_LOGIC), col('base_price') * lit(1.15))  # 15% increase
    .otherwise(col('base_price'))  # dead branch — never reached
)

result.explain('extended')
# In the Optimized Logical Plan, the CASE WHEN is folded to:
# (base_price * 1.15) AS price
# The otherwise branch is eliminated — no runtime evaluation of the condition
```

**Expected Outcome:**
Catalyst replaces `when(true, expr).otherwise(other)` with just `expr` in the optimized plan.
No runtime branching overhead. Dead code is pruned at optimization time.

**Exam Sub-topic:** Constant folding; Catalyst logical optimization; explain('extended') to see optimized plan
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| Catalyst phases | Analysis → Logical Optimization → Physical Planning → Code Generation |
| `explain()` | Physical plan only; read **bottom to top** |
| `explain('extended')` | All 4 plans (Parsed, Analyzed, Optimized, Physical) |
| `explain('formatted')` | Plan + node details; shows `PushedFilters`, `ReadSchema` |
| Predicate pushdown | Filter moved to data source scan — reduces rows read |
| Column pruning | Only referenced columns read from Parquet/ORC — reduces I/O |
| Constant folding | `1 + 1` → `2` at plan time; dead branches eliminated |
| BroadcastHashJoin | Small table broadcast to all executors; **no shuffle of large table** |
| SortMergeJoin | Both tables shuffled + sorted; default for large-large joins |
| `broadcast()` hint | Forces BHJ regardless of table size |
| `autoBroadcastJoinThreshold` | Default 10 MB; `-1` disables auto broadcast |
| AQE | Re-optimizes at runtime; **enabled by default in Spark 3.2+** |
| AQE coalesce partitions | Merges small post-shuffle partitions based on actual data |
| AQE SMJ→BHJ conversion | Converts sort-merge to broadcast join if one side turns out small |
| `spark.sql.shuffle.partitions` | Default 200; often too high for small data |
| Partition pruning | Filter on partition column → only matching directories read |
| `PartitionFilters` in plan | Partition pruning is happening |
| `DataFilters` in plan | Row-level filter (no partition skip) |
| `AdaptiveSparkPlan` | Root node when AQE enabled — plan may change at runtime |
| `*(n)` in plan | WholeStageCodeGen stage — JVM bytecode compiled |
| `Exchange` in plan | Shuffle operation between stages |

---

## Practice Q&A

<details>
<summary>Q1: You see Exchange hashpartitioning in the explain() output. What does this mean?</summary>

**A:** `Exchange hashpartitioning(key, N)` means a **shuffle** is occurring. Data is being repartitioned across the cluster by hashing the specified key into N partitions. This creates a stage boundary. Each `Exchange` node in the plan represents a shuffle — an expensive operation involving disk I/O and network transfer. Minimising the number of `Exchange` nodes improves performance.
</details>

<details>
<summary>Q2: What is the difference between PushedFilters and DataFilters in a formatted explain plan?</summary>

**A:**
- `PushedFilters`: Filters that have been **pushed into the data source** (e.g., Parquet reader). The data source applies these filters while reading, meaning matching rows are never loaded into Spark memory. This is predicate pushdown.
- `DataFilters`: Row-level filters applied by Spark **after** reading data from the source. The rows are loaded into memory first, then filtered.
- `PartitionFilters`: Applied at the **file system level** to skip entire partition directories — the most powerful form of pruning.

For maximum performance: you want filters to appear as `PartitionFilters` → `PushedFilters` → `DataFilters` (in that order of preference).
</details>

<details>
<summary>Q3: Your job has spark.sql.shuffle.partitions=200 but only 5 categories in the groupBy. What happens without AQE?</summary>

**A:** Without AQE, Spark creates **200 output partitions** after the shuffle regardless of how much data they contain. With only 5 categories, most of the 200 partitions will be **empty or tiny** (near zero rows). This causes:
1. Task scheduling overhead for 200 tasks
2. Many tasks that finish almost instantly (scheduling cost > compute cost)
3. Tiny output files if writing to storage

**Fix options:**
- Enable AQE (coalesces tiny partitions automatically)
- Reduce `spark.sql.shuffle.partitions` to match the actual data size
- Use `coalesce()` after the aggregation
</details>

<details>
<summary>Q4: What does explain('extended') show that explain() does not?</summary>

**A:** `explain('extended')` shows all **4 plan stages**:
1. **Parsed Logical Plan** — raw plan before name/type resolution
2. **Analyzed Logical Plan** — after resolving column names and types against catalog
3. **Optimized Logical Plan** — after Catalyst rule-based optimizations (predicate pushdown, constant folding, etc.)
4. **Physical Plan** — the actual execution plan with chosen strategies (BHJ vs SMJ, etc.)

Plain `explain()` or `explain('simple')` shows only the Physical Plan.
Comparing the Optimized Logical Plan to the Analyzed Plan shows what Catalyst changed.
</details>

<details>
<summary>Q5: When does AQE switch a SortMergeJoin to a BroadcastHashJoin?</summary>

**A:** AQE can convert a `SortMergeJoin` to a `BroadcastHashJoin` at **runtime** when:
1. AQE is enabled (`spark.sql.adaptive.enabled = true`)
2. After executing the build side of the join, its actual materialized size is below `spark.sql.adaptive.autoBroadcastJoinThreshold` (default 30 MB)
3. The original plan chose SMJ because the table's **estimated** size was above the static `autoBroadcastJoinThreshold` at planning time (e.g., it was a subquery result that Catalyst couldn't size statically)

This is a key AQE advantage: static planning uses estimates; AQE uses **actual runtime statistics**.
</details>